In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica por variáveis (Linear Regression por features)
+ Aplicação global em toda a base (sem e com falha)
+ Classificação de falhas (RandomForestClassifier)
+ Split térmico automático intercalado (sem overlap)
+ RF regularizado para evitar overfitting
Autor: Luiz Eduardo Abdala José
"""

import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

# ========= PARÂMETROS =========
ARQ_BASE = "base-completo--.pkl"
REF_TEMP = 20
FREQ_MIN_KHZ = 30
FREQ_MAX_KHZ = 70
SMOOTH_WIN = 5

CAPS = dict(gain_frac=0.60, offset_frac=0.60, tilt_frac=0.40)
TAU_MAX_FRAC = 0.025
ANCHOR_TO_REF_ENDS = True

# Random Forest regularizado (igual ao do Park)
RF_CLASSIF_PARAMS = dict(
    n_estimators=150,
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=6,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# ========= FUNÇÕES AUXILIARES =========
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs)[order]

def moving_average(arr, win):
    if win<=1 or win%2==0: return arr
    pad = win//2
    arr_pad = np.pad(arr, (pad, pad), mode='edge')
    kernel = np.ones(win)/win
    smooth = np.convolve(arr_pad, kernel, mode='valid')
    if len(smooth) > len(arr): smooth = smooth[:len(arr)]
    elif len(smooth) < len(arr): smooth = np.pad(smooth, (0, len(arr)-len(smooth)), mode='edge')
    return smooth

def shift_interp(x_row, fhz, tau_hz):
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

def slope_over_band(f, x):
    return float((x[-1] - x[0]) / (f[-1] - f[0] + 1e-12))

def energy_weighted_centroid(f, x):
    xm = np.asarray(x, float)
    w = xm * xm
    den = float(np.trapezoid(w, f))
    if den <= 1e-18: return float(np.mean(f))
    num = float(np.trapezoid(f * w, f))
    return num / den

# ======= EXTRAÇÃO DE FEATURES ==========
def compute_features(X, f):
    X = np.asarray(X, float); n, m = X.shape
    out = []
    for i in range(n):
        x = X[i]
        mean = float(np.mean(x))
        std  = float(np.std(x))
        amp  = float(x.max() - x.min())
        slope = slope_over_band(f, x)
        centroid = energy_weighted_centroid(f, x)
        out.append([mean, std, amp, slope, centroid])
    cols = ["mean", "std", "amp", "slope", "centroid"]
    return np.array(out, float), cols

# ======= AJUSTE LINEAR DAS FEATURES ==========
def fit_feature_vs_temp_models(F, T, feat_names):
    models = {}
    T = np.asarray(T, float).reshape(-1,1)
    for j, name in enumerate(feat_names):
        lr = LinearRegression()
        lr.fit(T, F[:, j])
        models[name] = lr
    return models

def feature_targets_at_ref(models, ref_temp=REF_TEMP):
    Tref = np.array([[ref_temp]])
    return {name: float(lr.predict(Tref)[0]) for name, lr in models.items()}

# ======= COMPENSAÇÃO ==========
def apply_compensation_by_features(x, f, targets, caps, y_ref=None):
    x = x.copy()
    mean_t = targets["mean"]; amp_t = targets["amp"]; slope_t = targets["slope"]
    centroid_t = targets["centroid"]

    mean_x = float(np.mean(x)); amp_x = float(x.max() - x.min()); slope_x = slope_over_band(f, x)

    # Offset
    offset = mean_t - mean_x
    offset_cap = caps["offset_frac"] * max(1e-9, amp_x)
    offset = float(np.clip(offset, -offset_cap, offset_cap))
    x = x + offset

    # Ganho
    gain = 1.0 if amp_x <= 1e-9 else float(amp_t / amp_x)
    gmin = 1.0 - caps["gain_frac"]; gmax = 1.0 + caps["gain_frac"]
    gain = float(np.clip(gain, gmin, gmax))
    x = mean_t + gain * (x - mean_t)

    # Inclinação
    delta_slope = slope_t - slope_x
    u = np.linspace(-0.5, 0.5, len(x))
    df = (f[-1] - f[0] + 1e-12)
    tilt_signal = (delta_slope * df) * u
    tilt_cap = caps["tilt_frac"] * max(1e-9, amp_x)
    tilt_signal = np.clip(tilt_signal, -tilt_cap, tilt_cap)
    x = x + tilt_signal

    # Centroide espectral
    cent_x = energy_weighted_centroid(f, x)
    delta_c = centroid_t - cent_x
    tau_max = TAU_MAX_FRAC * (f[-1] - f[0])
    tau = float(np.clip(delta_c, -tau_max, tau_max))
    if abs(tau) > 1e-12:
        x = shift_interp(x, f, tau)

    # Ancoragem
    if ANCHOR_TO_REF_ENDS and (y_ref is not None):
        e0 = x[0] - y_ref[0]; e1 = x[-1] - y_ref[-1]
        corr = np.linspace(e0, e1, len(x))
        x = x - corr

    return x

def compensate_all(df, fcols, fhz, ref_temp=REF_TEMP):
    print("🔹 Treinando regressões lineares das features (somente sem falha)...")
    df_sem = df[df["falha"] == 0].copy()
    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)
    F_sem, feat_names = compute_features(X_sem, fhz)
    feat_models = fit_feature_vs_temp_models(F_sem, T_sem, feat_names)
    targets = feature_targets_at_ref(feat_models, ref_temp)

    pool_ref = df_sem.loc[np.isclose(df_sem["temperatura_c"], ref_temp), fcols].to_numpy(float)
    if len(pool_ref) == 0:
        print("⚠️ Nenhum dado exato a 20°C sem falha encontrado, usando média global sem falha.")
        y_ref = np.median(X_sem, axis=0)
    else:
        y_ref = np.median(pool_ref, axis=0)

    print("🔹 Aplicando compensação em toda a base...")
    X_all = df[fcols].to_numpy(float)
    Y = np.zeros_like(X_all)
    for i in range(X_all.shape[0]):
        y = apply_compensation_by_features(X_all[i], fhz, targets, CAPS, y_ref=y_ref)
        if SMOOTH_WIN > 1 and SMOOTH_WIN % 2 == 1:
            y = moving_average(y, SMOOTH_WIN)
        Y[i] = y

    df_comp = df.copy()
    df_comp[fcols] = Y
    print("✅ Compensação concluída.")
    return df_comp, y_ref

# ========= ETAPA 1 – CARREGAMENTO =========
print("🔹 Carregando base completa...")
df = pd.read_pickle(ARQ_BASE)
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz_khz = fhz / 1e3
print(f"Nº amostras: {len(df)} | Nº features: {len(fcols)}")

# ========= ETAPA 2 – COMPENSAÇÃO =========
df_comp, y_ref = compensate_all(df, fcols, fhz, REF_TEMP)

# ========= ETAPA 3 – SPLIT TÉRMICO INTERCALADO =========
temps_all = sorted(df_comp["temperatura_c"].unique())
temps_train = temps_all[::2]
temps_test  = temps_all[1::2]

print(f"\nTemperaturas treino: {temps_train}")
print(f"Temperaturas teste:  {temps_test}")

df_train = df_comp[df_comp["temperatura_c"].isin(temps_train)].copy()
df_test  = df_comp[df_comp["temperatura_c"].isin(temps_test)].copy()

X_train = df_train[fcols].to_numpy(float)
y_train = df_train["falha"].to_numpy(int)
X_test  = df_test[fcols].to_numpy(float)
y_test  = df_test["falha"].to_numpy(int)

print(f"Amostras treino: {len(X_train)} | teste: {len(X_test)}")

# ========= ETAPA 4 – CLASSIFICAÇÃO =========
print("\n🔹 Treinando RandomForestClassifier (regularizado)...")
clf = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n== RESULTADOS RANDOM FOREST (LinearRegression por features, split térmico intercalado, RF regularizado) ==")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

# ========= ETAPA 5 – PLOT =========
print("\n🔹 Gerando gráfico de exemplo...")
idx_show = df_test.index[10] if len(df_test) > 10 else df_test.index[0]

plt.figure(figsize=(9, 5))
plt.plot(fhz_khz, y_ref, '--', c='black', lw=1.2, label=f"Referência {REF_TEMP}°C (sem falha)")
plt.plot(fhz_khz, df.loc[idx_show, fcols], c='tab:red', alpha=0.6,
         label=f"Original {df.loc[idx_show,'temperatura_c']}°C (falha={df.loc[idx_show,'falha']})")
plt.plot(fhz_khz, df_comp.loc[idx_show, fcols], c='tab:blue', lw=2,
         label=f"Compensado Linear {df.loc[idx_show,'temperatura_c']}°C")
plt.title(f"Compensação Linear — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.legend()
plt.tight_layout()
plt.grid(alpha=0.3)
plt.show()

print("\n✅ Execução completa (Linear + split térmico automático intercalado).")


In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.spatial.distance import cosine
from math import acos, degrees

def calc_metrics(y_true, y_pred):
    """Calcula todas as métricas entre curvas"""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0,1]
    # SAM (Spectral Angle Mapper)
    sam_rad = acos(np.clip(np.dot(y_true, y_pred) /
                           (np.linalg.norm(y_true) * np.linalg.norm(y_pred) + 1e-12), -1, 1))
    sam_deg = degrees(sam_rad)
    nrmse = rmse / (y_true.max() - y_true.min() + 1e-12)
    rmsd = np.sqrt(np.mean((y_true - y_pred - np.mean(y_true - y_pred))**2))
    ccdm = 1 - corr
    return dict(R2=r2, RMSE=rmse, MAE=mae, Corr=corr,
                SAM_deg=sam_deg, NRMSE=nrmse, RMSD=rmsd, CCDM=ccdm)

# ===== Exemplo de uso =====
# y_ref: referência 20°C (ex: y_ref do código principal)
# X_orig: curva original (ex: df.loc[idx_show, fcols])
# X_comp: curva compensada (ex: df_comp.loc[idx_show, fcols])

y_ref_vec = y_ref
X_orig = df.loc[idx_show, fcols].to_numpy(float)
X_comp = df_comp.loc[idx_show, fcols].to_numpy(float)

print("\n== MÉTRICAS ORIGINAL vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_orig))

print("\n== MÉTRICAS COMPENSADO vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_comp))
